# 🤖 Construindo um Agente de IA: Manual do Aluno Braz Cubas

---

Bem-vindo(a)! Neste notebook você vai **construir passo a passo** um agente de Inteligência Artificial capaz de responder dúvidas sobre o Manual do Aluno.

### Como funciona esta aula?

Cada etapa segue o mesmo ritual:

1. 📖 **Leia** a explicação do conceito
2. 🤔 **Responda** a pergunta na célula de código
3. ▶️ **Execute** a célula para ver se acertou
4. ⚙️ **Execute** o código real da etapa

> ⚠️ **Não pule etapas!** Cada conceito é necessário para entender o próximo.

---

### O que vamos construir?

```
  [PDF - Manual do Aluno]
          |
          v
  [ETAPA 1] PyPDFLoader      --> Lê e carrega o PDF
          |
          v
  [ETAPA 2] Text Splitter    --> Divide o texto em pedaços (chunks)
          |
          v
  [ETAPA 3] Embeddings       --> Transforma texto em números
          |
          v
  [ETAPA 4] ChromaDB         --> Guarda os números num banco vetorial
          |
          v
  [ETAPA 5] Retriever        --> Busca os trechos mais relevantes
          |
          v
  [ETAPA 6] LLM Groq         --> Gera a resposta final
          |
          v
  [ETAPA 7] Interface Gradio --> Chat para o usuário
```

---
## ⚙️ Preparação: Instalação das bibliotecas

Antes de começar, precisamos instalar todas as ferramentas que vamos usar.

> 📦 **Biblioteca** é como um "pacote de ferramentas prontas" que alguém já criou para que a gente não precise criar tudo do zero. Com `pip install`, baixamos e instalamos essas ferramentas no nosso ambiente.

### 🤔 Pergunta 0.1 — Antes de instalar

Na célula abaixo, complete a variável `resposta` com a letra correta e execute para ver o resultado.

**Para que serve o comando `!pip install` no Google Colab?**

- `a` → Para criar um arquivo novo no computador
- `b` → Para baixar e instalar bibliotecas Python que vamos usar no código
- `c` → Para executar o programa principal
- `d` → Para conectar ao banco de dados

In [ ]:
resposta = "b"  # Digite a letra entre as aspas: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
O comando !pip install baixa e instala bibliotecas Python.
No nosso caso, instalamos ferramentas como LangChain (para
orquestrar a IA), ChromaDB (banco de dados vetorial), Groq
(acesso ao modelo de linguagem) e Gradio (interface visual).
Sem essas instalações, o código não funcionaria!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' com uma letra antes de executar.")
else:
    print(f"❌ Não é bem isso. Tente novamente! Dica: o ! antes de pip indica que é um comando do sistema operacional, não Python puro.")

In [ ]:
# ⚙️ CÓDIGO REAL — Execute após responder a pergunta acima
!pip install -q \
  langchain==0.2.1 \
  langchain-core==0.2.43 \
  langchain-community==0.2.4 \
  langchain-groq==0.1.6 \
  langchain-text-splitters==0.2.4 \
  chromadb \
  pypdf==4.2.0 \
  sentence-transformers==2.7.0 \
  gradio \
  numpy==1.26.4
print("✅ Todas as bibliotecas instaladas com sucesso!")

---
## ⚙️ Preparação: Configuração da API Key

Para usar o modelo de linguagem da Groq, precisamos de uma **API Key** — uma espécie de "senha de acesso" ao serviço.

> 🔑 **API Key** é um código secreto que identifica quem está fazendo requisições a um serviço. É como um crachá de acesso: sem ele, o serviço não sabe quem você é e não deixa entrar.

> ⚠️ **Antes de executar o código abaixo:** Acesse [console.groq.com](https://console.groq.com), crie sua conta gratuita e gere uma API Key. Em seguida, no Google Colab, clique no ícone de chave 🔑 (Secrets) no menu lateral esquerdo e adicione a chave com o nome exato: `GROQ_KEY`.

### 🤔 Pergunta 0.2 — API Key

**Por que NÃO devemos escrever a API Key diretamente no código, assim:**
```python
os.environ["GROQ_KEY"] = "minha-chave-super-secreta-123"
```

Complete a variável `resposta` com `"verdadeiro"` ou `"falso"`:

**Afirmação:** *"Colocar a chave diretamente no código é perigoso porque, se alguém ver ou compartilhar o notebook, a chave fica exposta e qualquer pessoa pode usá-la."*

In [ ]:
resposta = "verdadeiro"  # Digite "verdadeiro" ou "falso"

# ---- Não altere o código abaixo ----
gabarito = "verdadeiro"
explicacao = """
Exatamente! Por isso usamos os 'Secrets' do Colab.
O código userdata.get('GROQ_KEY') lê a chave de um lugar
seguro, sem que ela apareça no notebook. Assim, você pode
compartilhar o código com colegas sem expor sua senha.
É uma boa prática de segurança muito usada em projetos reais!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! Pense: o que aconteceria se você postasse o notebook no GitHub com a chave visível?")

In [ ]:
# ⚙️ CÓDIGO REAL — Imports e configuração
import os
from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

os.environ["GROQ_KEY"] = userdata.get("GROQ_KEY")
print("✅ API Key do Groq carregada com segurança!")

---
## 📄 ETAPA 1 — Carregando o PDF

O primeiro passo é **ler o PDF** do Manual do Aluno e transformá-lo em texto que o Python consegue manipular.

Usamos a biblioteca `PyPDFLoader` para isso. Ela abre o arquivo PDF, página por página, e extrai o texto de cada uma.

```python
loader = PyPDFLoader("caminho/do/arquivo.pdf")
documents = loader.load()
```

Após executar esse código, a variável `documents` será uma **lista** — cada elemento da lista corresponde a uma página do PDF.

> 📋 **Lista** em Python é uma coleção ordenada de itens. Exemplo: `["maçã", "banana", "uva"]`. Podemos saber quantos itens tem com `len(lista)`.

### 🤔 Pergunta 1.1 — Listas e len()

**Se o Manual do Aluno tem 30 páginas e cada página virou um item da lista `documents`, qual será o resultado de `len(documents)`?**

- `a` → 1
- `b` → 30
- `c` → 300
- `d` → Depende do tamanho do texto

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
len() retorna a quantidade de itens em uma lista.
Como cada página do PDF vira um item na lista 'documents',
um PDF de 30 páginas resultará em len(documents) == 30.
Isso nos permite saber exatamente quantas páginas foram carregadas.
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Não é isso. Dica: len() conta itens na lista, e cada página = 1 item.")

### 🤔 Pergunta 1.2 — Por que carregar o PDF?

**Pesquise ou reflita:** Por que não podemos simplesmente perguntar ao ChatGPT ou ao LLaMA sobre o Manual do Aluno da Braz Cubas sem carregá-lo primeiro?

Complete com `"a"` ou `"b"`:

- `a` → Porque a IA já foi treinada com o Manual do Aluno e pode responder qualquer pergunta sobre ele
- `b` → Porque o Manual do Aluno é um documento interno e específico que a IA nunca viu durante o treinamento, então ela inventaria respostas erradas

In [ ]:
resposta = ""  # Digite "a" ou "b"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Isso se chama 'alucinação' da IA: quando o modelo inventa
uma resposta que parece verdadeira, mas é falsa.
Documentos internos (manuais, regulamentos, contratos)
nunca fazem parte do treinamento dos modelos de IA.
Por isso precisamos do RAG: fornecemos o documento diretamente
para que a IA baseie as respostas SOMENTE no que está escrito.
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! Pense: a Braz Cubas publicou o Manual do Aluno na internet para a IA aprender? Provavelmente não...")

In [ ]:
# ⚙️ CÓDIGO REAL — Carregando o PDF
PDF_PATH = "/content/Manual do Aluno.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"📄 Páginas carregadas: {len(documents)}")
print(f"\n🔍 Prévia do texto da primeira página:")
print(documents[1].page_content[:200], "...")

---
## ✂️ ETAPA 2 — Dividindo o texto em pedaços (Chunks)

O texto completo do Manual tem muitas páginas. Se mandássemos tudo de uma vez para a IA, seria como tentar ler um livro inteiro em 1 segundo — impossível!

Por isso dividimos o texto em **chunks** (pedaços menores). Cada chunk tem aproximadamente 300 caracteres, com um pequeno trecho de sobreposição (`chunk_overlap`) para não perder contexto entre os pedaços.

```
Texto completo: |----página 1----|----página 2----|----página 3----|
                         ↓  dividir
Chunks:         |chunk1|chunk2|chunk3|chunk4|chunk5|chunk6|chunk7|
```

> 🔄 **chunk_overlap** é como um "grampo" entre pedaços: o final de um chunk aparece também no início do próximo, para que frases não fiquem cortadas ao meio.

### 🤔 Pergunta 2.1 — Chunks

**Por que dividir o texto em chunks menores antes de processar?**

- `a` → Para que o código fique mais organizado visualmente
- `b` → Porque modelos de IA têm um limite de texto que conseguem processar de uma vez, e chunks menores facilitam a busca precisa pelo trecho relevante
- `c` → Para economizar espaço no Google Drive
- `d` → Porque o PDF não consegue ser lido inteiro de uma vez

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Os modelos de IA têm um 'contexto máximo' — um limite de
quantos caracteres conseguem processar por vez.
Além disso, com chunks menores, quando o usuário faz uma
pergunta, conseguimos encontrar EXATAMENTE o trecho do
manual que fala sobre aquele assunto, em vez de jogar
o documento inteiro para a IA interpretar.
É mais preciso, mais rápido e mais barato!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Não é isso. Pense: se você quer saber sobre frequência, faz sentido passar o manual inteiro ou só os trechos sobre frequência?")

### 🤔 Pergunta 2.2 — chunk_overlap

Imagine que o texto seja: `"O aluno pode ter no máximo 25% de faltas por disciplina."`

Se dividirmos exatamente no meio, um chunk termina em `"25%"` e o próximo começa em `"de faltas"` — perdendo o contexto!

**O que o `chunk_overlap=40` faz para resolver isso?**

Complete com `"a"` ou `"b"`:
- `a` → Ignora os últimos 40 caracteres de cada chunk
- `b` → Repete os últimos 40 caracteres de um chunk no início do próximo, mantendo o contexto

In [ ]:
resposta = ""  # Digite "a" ou "b"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
O overlap (sobreposição) garante que informações que
ficam na 'fronteira' entre dois chunks não se percam.
É como páginas de um livro que repetem a última linha
da página anterior — você nunca perde o fio da história!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! 'Overlap' significa sobreposição — algo que aparece em dois lugares ao mesmo tempo.")

In [ ]:
# ⚙️ CÓDIGO REAL — Dividindo o texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=40
)
chunks = text_splitter.split_documents(documents)
print(f"✂️  Chunks gerados: {len(chunks)}")
print(f"\n🔍 Exemplo do chunk 10:")
print(chunks[10].page_content)
print(f"\n📏 Tamanho deste chunk: {len(chunks[10].page_content)} caracteres")

---
## 🔢 ETAPA 3 — Embeddings: transformando texto em números

Computadores não entendem palavras — eles entendem **números**. Para que o computador consiga comparar textos e encontrar os mais parecidos, precisamos transformar cada chunk em uma lista de números chamada **vetor** (ou **embedding**).

Usamos um modelo gratuito do HuggingFace para isso:
```python
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
```

Este modelo transforma qualquer frase em um vetor de **384 números**. Textos com significados parecidos geram vetores numericamente próximos!

```
"Quantas faltas posso ter?"  → [0.23, -0.14, 0.87, 0.05, ...]
"Limite de faltas por aula" → [0.25, -0.12, 0.85, 0.04, ...]  ← próximos!
"Receita de bolo de cenoura" → [-0.91, 0.43, -0.22, 0.78, ...] ← distante!
```

### 🤔 Pergunta 3.1 — O que são embeddings?

**Por que precisamos transformar texto em números (embeddings)?**

- `a` → Para comprimir o arquivo e ocupar menos espaço no disco
- `b` → Porque é mais fácil de ler para humanos
- `c` → Para que o computador consiga calcular matematicamente a semelhança entre textos e encontrar os trechos mais relevantes para uma pergunta
- `d` → Para traduzir o texto para inglês automaticamente

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "c"
explicacao = """
Com embeddings, podemos usar matemática (distância entre vetores)
para medir o quão parecidos dois textos são em SIGNIFICADO.
Isso é o coração do sistema RAG: quando você faz uma pergunta,
ela também vira um vetor, e buscamos os chunks cujos vetores
são mais próximos (matematicamente) ao vetor da sua pergunta.
É busca por significado, não por palavras exatas!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Não é isso. Pense: como o computador saberia que 'falta' e 'ausência' têm o mesmo significado sem transformar em números?")

### 🤔 Pergunta 3.2 — Pesquisa rápida

Pesquise na internet (pode usar o Google!) e responda:

**O modelo `all-MiniLM-L6-v2` que usamos é gratuito e roda localmente. O que isso significa na prática para o aluno?**

- `a` → Significa que precisamos pagar para usar depois de 30 dias
- `b` → Significa que não gastamos dinheiro e a IA não precisa da internet para gerar os vetores
- `c` → Significa que só funciona em computadores com placa de vídeo potente
- `d` → Significa que os resultados são menos precisos que modelos pagos

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Exatamente! O modelo é baixado uma vez e roda diretamente
na máquina do Colab (localmente), sem precisar chamar
nenhuma API externa. Isso significa:
  ✅ Zero custo (gratuito)
  ✅ Privacidade (o texto não sai do seu ambiente)
  ✅ Velocidade (sem latência de rede)
Compare com a alternativa paga da OpenAI, que cobra por
cada chamada de embedding!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! Dica: 'roda localmente' = roda na própria máquina, sem precisar da internet após o download.")

In [ ]:
# ⚙️ CÓDIGO REAL — Criando o modelo de embeddings
# (O modelo será baixado automaticamente na primeira execução — aguarde!)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Demonstração: veja como um texto vira números!
exemplo = "Quantas faltas posso ter?"
vetor = embeddings.embed_query(exemplo)
print(f"Texto: '{exemplo}'")
print(f"Tamanho do vetor: {len(vetor)} números")
print(f"Primeiros 5 valores: {[round(v, 4) for v in vetor[:5]]}")
print("\n✅ Modelo de embeddings pronto!")

---
## 🗄️ ETAPA 4 — ChromaDB: o banco de dados vetorial

Agora que temos os chunks e o modelo de embeddings, precisamos **armazenar tudo** em um lugar onde possamos fazer buscas rápidas. Para isso, usamos o **ChromaDB**, um banco de dados especializado em vetores.

O processo é:
1. Pegar cada chunk de texto
2. Transformar em vetor (embedding)
3. Salvar o par `(texto, vetor)` no ChromaDB

```
ChromaDB armazena:
┌─────────────────────────────────────────────────────────┐
│  texto: "O limite de faltas é 25%..."                   │
│  vetor: [0.23, -0.14, 0.87, 0.05, ...]                 │
│  página: 11                                             │
├─────────────────────────────────────────────────────────┤
│  texto: "A nota mínima para aprovação é 6.0..."        │
│  vetor: [0.11, 0.55, -0.32, 0.88, ...]                 │
│  página: 14                                             │
└─────────────────────────────────────────────────────────┘
```

> 🗃️ **Banco de dados vetorial** é diferente de bancos de dados tradicionais (como Excel ou MySQL). Em vez de buscar por palavras exatas, ele busca por **similaridade de significado** usando a distância matemática entre vetores.

### 🤔 Pergunta 4.1 — Banco vetorial vs. banco tradicional

Imagine que o aluno pergunta: **"Posso faltar muito?"**

No Manual, o texto relevante diz: **"O limite de frequência é de 75%"**.

Note que as palavras são completamente diferentes! O aluno disse "faltar" e o manual diz "frequência".

**Qual tipo de banco de dados consegue encontrar esse trecho mesmo com palavras diferentes?**

- `a` → Um banco tradicional (como Excel), porque ele busca palavra por palavra
- `b` → Um banco vetorial (como ChromaDB), porque ele busca por significado, não por palavras exatas

In [ ]:
resposta = ""  # Digite "a" ou "b"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Um banco tradicional buscaria pela palavra 'faltar' e
não encontraria nada (pois o manual usa 'frequência').
O banco vetorial transforma ambas as frases em vetores
e percebe que os vetores de 'Posso faltar muito?' e
'O limite de frequência é 75%' são matematicamente
próximos — ou seja, têm significados relacionados.
Por isso o RAG funciona tão bem!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Pense: se você buscar 'faltar' no Word (Ctrl+F) num documento que só fala em 'frequência', vai encontrar?")

### 🤔 Pergunta 4.2 — persist_directory

No código, usamos `persist_directory=CHROMA_DIR`. Isso salva o banco no disco do Colab.

**Por que isso é útil?**

- `a` → Para que não precisemos reprocessar todos os chunks toda vez que reiniciarmos o código
- `b` → Para que o banco fique mais rápido durante a busca
- `c` → Para comprimir os vetores e economizar memória
- `d` → Não tem utilidade, é apenas uma boa prática

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "a"
explicacao = """
Sem persistência, toda vez que você abrisse o Colab precisaria
reprocessar todos os 230 chunks — o que leva alguns minutos.
Com o banco salvo em disco, podemos carregá-lo instantaneamente
em execuções futuras. Em um sistema real com milhares de documentos,
isso faz uma diferença enorme!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! 'Persist' em inglês significa 'persistir/guardar'. O que seria útil guardar para não precisar refazer?")

In [ ]:
# ⚙️ CÓDIGO REAL — Criando o banco vetorial ChromaDB
import shutil

CHROMA_DIR = "/content/chroma_manual_aluno_v1"

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
print(f"🗑️  Diretório anterior removido")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)
print(f"✅ Banco vetorial criado com {len(chunks)} chunks em: {CHROMA_DIR}")

---
## 🔍 ETAPA 5 — Retriever: o mecanismo de busca

O **Retriever** é o componente que recebe a pergunta do usuário e **busca no ChromaDB** os chunks mais relevantes.

O processo acontece assim:

```
Pergunta do usuário: "Quantas faltas posso ter?"
         ↓
  Transforma em vetor: [0.22, -0.13, 0.86, ...]
         ↓
  Busca no ChromaDB os 3 vetores mais próximos
         ↓
  Retorna os 3 chunks de texto correspondentes
```

Configuramos `k=3`, ou seja, o retriever sempre devolve os **3 trechos mais relevantes**.

### 🤔 Pergunta 5.1 — O papel do Retriever

**Qual é a diferença entre o Retriever e o LLM (modelo de linguagem) nesta arquitetura?**

- `a` → São a mesma coisa, só têm nomes diferentes
- `b` → O Retriever BUSCA os trechos relevantes no banco de dados; o LLM INTERPRETA esses trechos e gera a resposta em linguagem natural
- `c` → O Retriever gera a resposta; o LLM faz a busca
- `d` → O Retriever lida com imagens; o LLM lida com texto

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
É uma divisão clara de responsabilidades:
  🔍 RETRIEVER = Bibliotecário: sabe onde estão os livros
                  e pega os mais relevantes para você
  🧠 LLM       = Professor: lê os trechos que o bibliotecário
                  trouxe e explica em linguagem clara

O LLM SÓ responde com base no que o Retriever trouxe.
Isso garante que as respostas sejam baseadas no Manual,
e não em 'achismos' do modelo!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Não é isso. Pense em dois personagens: um que encontra a informação e outro que a explica.")

### 🤔 Pergunta 5.2 — O parâmetro k

Configuramos `search_kwargs={"k": 3}`, ou seja, o retriever retorna 3 chunks.

**O que aconteceria se aumentássemos para `k=50`?**

- `a` → A resposta seria sempre melhor, pois teria mais informação
- `b` → Poderia sobrecarregar o LLM com trechos irrelevantes, deixar a resposta mais lenta e mais cara
- `c` → O código daria erro pois k não pode ser maior que 5
- `d` → Nada mudaria, o k não afeta o resultado

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Mais nem sempre é melhor! Com k=50, mandaríamos 50 trechos
para o LLM processar — muitos deles provavelmente irrelevantes.
Isso tornaria o prompt enorme, a resposta mais lenta,
e em modelos pagos, mais caro (pagamos por token/palavra).
k=3 é um bom equilíbrio: informação suficiente sem poluir
o contexto com ruído desnecessário.
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Pense: se você pede para um amigo buscar informações e ele traz 50 livros irrelevantes junto com os 3 certos, ajuda ou atrapalha?")

In [ ]:
# ⚙️ CÓDIGO REAL — Criando o Retriever e testando uma busca
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Demonstração: veja o que o retriever encontra para uma pergunta
print("🔍 Testando o Retriever com a pergunta: 'Quantas faltas posso ter?'\n")
resultados = retriever.get_relevant_documents("Quantas faltas posso ter?")
for i, doc in enumerate(resultados, 1):
    print(f"--- Trecho {i} (Página {doc.metadata.get('page', 'N/A')}) ---")
    print(doc.page_content)
    print()

---
## 🧠 ETAPA 6 — LLM Groq: gerando a resposta

Agora conectamos o **modelo de linguagem** (LLM) ao retriever para montar a cadeia RAG completa.

Usamos o **Groq** como provedor — ele oferece acesso gratuito ao modelo **LLaMA 3.1**, desenvolvido pela Meta (empresa do Facebook/Instagram).

O parâmetro `temperature` controla a **criatividade** do modelo:
```
temperature = 0.0  →  respostas muito factuais e repetíveis
temperature = 0.7  →  equilíbrio entre precisão e naturalidade
temperature = 1.0  →  respostas mais criativas e variadas
```

Para um chatbot de dúvidas acadêmicas, queremos respostas **precisas** — então `temperature=0.0` ou próximo disso é mais adequado!

> 🦙 **LLaMA** é um modelo de linguagem open-source criado pela Meta. "Open-source" significa que o código é público e qualquer empresa ou pesquisador pode usar. A Groq oferece acesso gratuito a ele via API.

### 🤔 Pergunta 6.1 — Temperature

Para um sistema de atendimento acadêmico que deve responder exatamente o que está no Manual do Aluno, **qual valor de temperature é mais adequado?**

- `a` → 1.0, para que as respostas sejam mais criativas e interessantes
- `b` → 0.0, para que as respostas sejam factuais e baseadas estritamente no contexto
- `c` → 0.5, sempre o meio-termo é o melhor
- `d` → O valor de temperature não importa para este caso

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
Para sistemas que precisam de precisão factual — como um
atendimento acadêmico, diagnóstico médico ou consultoria jurídica —
temperature baixa (0.0 ou próximo) é mais adequada.
O modelo vai se 'ater' ao contexto fornecido e não vai
'inventar' informações criativas.
Para geração de histórias, poemas ou brainstorming,
temperature alta seria melhor!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! Você preferiria que a IA fosse 'criativa' ao explicar quantas faltas você pode ter?")

### 🤔 Pergunta 6.2 — chain_type='stuff'

No código usamos `chain_type="stuff"`. Neste modo, o sistema **"enfia" (stuff)** todos os chunks recuperados em um único prompt para o LLM.

**O que isso significa na prática?**

- `a` → O LLM recebe a pergunta + os 3 trechos do Manual juntos e gera uma resposta baseada em tudo isso
- `b` → O LLM recebe apenas a pergunta, sem os trechos
- `c` → O LLM processa cada trecho separadamente e combina os resultados
- `d` → "stuff" significa que o código está desatualizado e funciona com erros

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "a"
explicacao = """
O prompt enviado ao LLM fica mais ou menos assim:

  'Use os trechos abaixo para responder a pergunta.

  Trecho 1: Por exemplo: Disciplina com 20h/a: 20x0,25...
  Trecho 2: Para saber o limite de faltas...
  Trecho 3: caso o estudante ultrapasse...

  Pergunta: Quantas faltas posso ter?'

O LLM então responde baseando-se SOMENTE nesses trechos.
É por isso que as respostas são precisas e não inventadas!
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Não é isso. 'Stuff' = enfiar tudo junto. Pense num recheio de pastel — tudo entra de uma vez!")

In [ ]:
# ⚙️ CÓDIGO REAL — Criando o LLM e a cadeia RAG completa
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    groq_api_key=os.environ["GROQ_KEY"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)
print("✅ Agente RAG completo e pronto para responder!")

---
## 🖥️ ETAPA 7 — Interface Gradio: o chatbot visual

Tudo que construímos até agora funciona nos bastidores. Agora vamos criar uma **interface visual** para que qualquer pessoa possa usar o agente sem precisar escrever código!

O **Gradio** é uma biblioteca Python que cria interfaces web com pouquíssimo código. Com `demo.launch(share=True)`, ele gera um link público que pode ser acessado em qualquer navegador.

```
Usuário digita pergunta no chat
         ↓
  Gradio captura e chama qa_chain.invoke(pergunta)
         ↓
  RAG busca trechos relevantes + LLM gera resposta
         ↓
  Gradio exibe pergunta + resposta + fontes na tela
```

### 🤔 Pergunta 7.1 — Para que serve o Gradio?

**Por que criamos uma interface com Gradio em vez de deixar o sistema funcionando só via código Python?**

- `a` → Porque o código Python para de funcionar sem interface visual
- `b` → Para que usuários sem conhecimento de programação possam interagir com o agente de IA de forma simples e intuitiva
- `c` → Para que o sistema fique mais rápido
- `d` → Porque o Gradio é obrigatório para usar o Groq

In [ ]:
resposta = ""  # Digite a letra: "a", "b", "c" ou "d"

# ---- Não altere o código abaixo ----
gabarito = "b"
explicacao = """
A interface é a 'porta de entrada' para qualquer usuário.
Um aluno da secretaria, um professor ou um calouro não sabe
Python — eles precisam de uma caixa de texto simples para
digitar a pergunta e ler a resposta.
Isso é o que o Gradio oferece: transformar código em produto!
Com share=True, o Gradio gera um link público por 72h,
acessível em qualquer dispositivo com internet.
"""
if resposta.strip().lower() == gabarito:
    print("✅ Correto! " + explicacao)
elif resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    print("❌ Revise! Pense: e se a secretaria quisesse usar este sistema? Precisaria aprender Python?")

### 🤔 Pergunta 7.2 — Revisão geral da arquitetura RAG

**Coloque as etapas da arquitetura RAG na ordem correta**, digitando os números de 1 a 7:

```
(  ) Retriever busca os 3 chunks mais relevantes no ChromaDB
(  ) LLM gera a resposta baseada nos chunks encontrados
(  ) PDF é carregado pelo PyPDFLoader
(  ) Embeddings transformam os chunks em vetores numéricos
(  ) Texto é dividido em chunks menores
(  ) Interface Gradio exibe a resposta ao usuário
(  ) Vetores são armazenados no ChromaDB
```

Complete a variável `resposta` com a sequência correta separada por vírgulas.
Exemplo de formato: `"3,1,5,2,6,4,7"`

In [ ]:
resposta = ""  # Ex: "3,1,5,2,6,4,7"

# ---- Não altere o código abaixo ----
# Ordem correta: PDF(3) > Chunks(5) > Embeddings(4) > ChromaDB(7) > Retriever(1) > LLM(2) > Gradio(6)
gabarito = ["3","5","4","7","1","2","6"]
pipeline = [
    "Retriever busca os 3 chunks mais relevantes no ChromaDB",
    "LLM gera a resposta baseada nos chunks encontrados",
    "PDF é carregado pelo PyPDFLoader",
    "Embeddings transformam os chunks em vetores numéricos",
    "Texto é dividido em chunks menores",
    "Interface Gradio exibe a resposta ao usuário",
    "Vetores são armazenados no ChromaDB",
]
if resposta == "":
    print("👆 Preencha a variável 'resposta' antes de executar.")
else:
    tentativa = [x.strip() for x in resposta.split(",")]
    if tentativa == gabarito:
        print("✅ Perfeito! Você dominou a arquitetura RAG completa!")
        print("\nPipeline correto:")
        for i, num in enumerate(gabarito, 1):
            print(f"  Etapa {i}: {pipeline[int(num)-1]}")
    else:
        print("❌ Não está certo ainda. Dica: pense na ordem lógica — primeiro você lê o documento, depois processa, depois usa!")
        print("Sua resposta:", tentativa)
        print("Gabarito:    ", gabarito)

In [ ]:
# ⚙️ CÓDIGO REAL — Interface Gradio (execute para abrir o chat!)
import gradio as gr

def responder_gradio(pergunta, historico):
    if not pergunta.strip():
        return historico, ""
    resultado = qa_chain.invoke(pergunta)
    resposta_llm = resultado["result"]
    fontes = resultado["source_documents"]
    linhas_fontes = ["\n---\n**Trechos do Manual utilizados:**"]
    for i, doc in enumerate(fontes, 1):
        pagina = doc.metadata.get("page", "N/A")
        linhas_fontes.append(
            f"**Trecho {i} | Página {pagina}:**\n{doc.page_content.strip()}"
        )
    resposta_completa = resposta_llm + "\n\n" + "\n\n".join(linhas_fontes)
    historico = historico + [[pergunta, resposta_completa]]
    return historico, ""

def limpar():
    return [], ""

with gr.Blocks(title="Manual do Aluno – Braz Cubas", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎓 Manual do Aluno – Braz Cubas
    Tire suas dúvidas acadêmicas! Respostas baseadas exclusivamente no **Manual do Aluno**.
    > Powered by **Groq LLaMA 3.1** + **HuggingFace Embeddings** (100% gratuito)
    """)
    chatbot = gr.Chatbot(label="Conversa", height=480, bubble_full_width=False)
    historico = gr.State([])
    with gr.Row():
        campo = gr.Textbox(placeholder="Digite sua dúvida e pressione Enter...", label="Sua pergunta", scale=8, lines=1)
        btn_enviar = gr.Button("Enviar", variant="primary", scale=1)
        btn_limpar = gr.Button("Limpar", scale=1)
    gr.Examples(
        examples=[
            "Quantas faltas posso ter por disciplina?",
            "Como é calculada a nota final?",
            "Como faço para trancar minha matrícula?",
            "O que é a Avaliação Final (AF)?",
        ],
        inputs=campo,
        label="Perguntas frequentes",
    )
    btn_enviar.click(fn=responder_gradio, inputs=[campo, historico], outputs=[chatbot, campo]).then(fn=lambda h: h, inputs=chatbot, outputs=historico)
    campo.submit(fn=responder_gradio, inputs=[campo, historico], outputs=[chatbot, campo]).then(fn=lambda h: h, inputs=chatbot, outputs=historico)
    btn_limpar.click(fn=limpar, outputs=[chatbot, campo]).then(fn=lambda: [], outputs=historico)

demo.launch(share=True)

---
## 🏆 Parabéns! Você concluiu o notebook!

Você construiu do zero um **Agente RAG completo**, entendendo cada peça da arquitetura:

| Etapa | Tecnologia | O que faz |
|-------|-----------|----------|
| 1 | PyPDFLoader | Lê o PDF e extrai o texto |
| 2 | RecursiveCharacterTextSplitter | Divide o texto em chunks |
| 3 | HuggingFace Embeddings | Transforma texto em vetores numéricos |
| 4 | ChromaDB | Armazena e indexa os vetores |
| 5 | Retriever | Busca os trechos mais relevantes |
| 6 | Groq LLaMA 3.1 | Gera a resposta em linguagem natural |
| 7 | Gradio | Interface visual para o usuário |

---

### 💭 Reflexão final

Antes de encerrar, responda mentalmente:

1. **O que é RAG?** (Retrieval Augmented Generation — geração aumentada por recuperação)
2. **Por que RAG é melhor que perguntar direto para a IA?** (Evita alucinações em documentos privados)
3. **Qual etapa você achou mais interessante? Por quê?**

---

> 🚀 **Desafio extra:** Substitua o `Manual do Aluno.pdf` por outro documento — um regulamento, uma lei, um contrato — e veja o agente responder sobre ele!